# Pair Trading Backtest
Pulls from `pair_trading_engine.py`. All config is in **Cell 2** — edit there and re-run.

In [ ]:
import sys
from pathlib import Path

# Make sure engine is importable (adjust if notebook is not in project root)
sys.path.insert(0, str(Path(".").resolve()))

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from IPython.display import display

from pair_trading_engine import PairTradingConfig, run

print("Engine loaded OK.")

## Configuration
Edit the hyperparameters here.

In [ ]:
cfg = PairTradingConfig(
    # ── Pair ─────────────────────────────────────────────────────────────────
    ts_code_a = "600519.SH",          # Stock A (Tushare ts_code)
    ts_code_b = "000858.SZ",          # Stock B

    # ── Windows ───────────────────────────────────────────────────────────────
    model_start = "20200101",         # In-sample start (OLS fit)
    model_end   = "20211231",         # In-sample end
    test_start  = "20220101",         # Out-of-sample start
    test_end    = "20231231",         # Out-of-sample end

    # ── Z-score thresholds ────────────────────────────────────────────────────
    entry_z     = 2.0,                # Open position when |z| > entry_z
    exit_z      = 0.0,                # Close position when |z| < exit_z
    stop_loss_z = 3.5,                # Emergency close when |z| > stop_loss_z

    # ── Direction ─────────────────────────────────────────────────────────────
    # Options: "long_short"  → trade both directions
    #          "long_only"   → only long spread (z < -entry_z)
    #          "short_only"  → only short spread (z > +entry_z)
    direction   = "long_short",

    # ── Rolling OLS (test window) ─────────────────────────────────────────────
    ols_window  = 60,                 # Trading days for rolling hedge ratio

    # ── Hedge ratio drift alerts ──────────────────────────────────────────────
    drift_alert_pct  = 0.20,         # Flag if β changes > 20% ...
    drift_alert_days = 10,           # ... within this many calendar days

    # ── Position sizing ───────────────────────────────────────────────────────
    # Options: "dollar_neutral"  → equal CNY on each leg (default)
    #          "fixed_notional"  → fixed CNY per trade (set fixed_notional)
    #          "vol_scaled"      → size inversely proportional to spread vol
    sizing          = "dollar_neutral",
    capital         = 1_000_000.0,   # Starting capital (CNY)
    fixed_notional  = 100_000.0,     # Used only when sizing = "fixed_notional"
    vol_window      = 20,            # Lookback days for vol_scaled sizing

    # ── Transaction costs ─────────────────────────────────────────────────────
    commission_rate = 0.0003,        # 0.03% each way (both legs)
    stamp_duty_rate = 0.001,         # 0.10% on SELL side only (A-share)

    # ── Database ──────────────────────────────────────────────────────────────
    db_path = Path("data/prices.sqlite"),
)

print("Config set. Run the next cell to execute backtest.")

## Run Backtest

In [ ]:
result = run(cfg)
print("Backtest complete.")
for w in result.warnings:
    print(" >", w)

## Summary Statistics

In [ ]:
stats_df = pd.DataFrame.from_dict(result.stats, orient="index", columns=["Value"])
stats_df.index.name = "Metric"

# Pretty labels
label_map = {
    "total_return_pct":          "Total Return (%)",
    "cagr_pct":                  "CAGR (%)",
    "sharpe_ratio":              "Sharpe Ratio (ann.)",
    "sortino_ratio":             "Sortino Ratio (ann.)",
    "max_drawdown_pct":          "Max Drawdown (%)",
    "calmar_ratio":              "Calmar Ratio",
    "n_round_trips":             "# Round-Trip Trades",
    "win_rate_pct":              "Win Rate (%)",
    "avg_round_trip_pnl_cny":    "Avg Round-Trip PnL (¥)",
    "total_transaction_costs_cny": "Total Transaction Costs (¥)",
    "start_value_cny":           "Starting Portfolio Value (¥)",
    "end_value_cny":             "Ending Portfolio Value (¥)",
    "in_sample_r2":              "In-Sample R²",
    "in_sample_beta":            "In-Sample β (hedge ratio)",
    "in_sample_alpha":           "In-Sample α",
    "model_spread_mean":         "Spread Mean (in-sample)",
    "model_spread_std":          "Spread Std (in-sample)",
    "n_drift_alerts":            "# Hedge Ratio Drift Alerts",
}
stats_df.index = [label_map.get(k, k) for k in stats_df.index]

display(stats_df.style
    .format("{:.4f}")
    .set_table_styles([{
        "selector": "th",
        "props": [("background-color", "#1a1a2e"), ("color", "white"), ("font-size", "13px")]
    }])
    .set_caption(f"Backtest: {cfg.ts_code_a} / {cfg.ts_code_b}  |  "
                 f"Test: {cfg.test_start} → {cfg.test_end}"))

## Hedge Ratio Drift Alerts

In [ ]:
if result.drift_alerts:
    alerts_df = pd.DataFrame(result.drift_alerts)
    display(alerts_df)
else:
    print("No hedge ratio drift alerts triggered.")

## Chart 1 — Equity Curve

In [ ]:
eq = result.equity_curve

fig = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    row_heights=[0.7, 0.3],
    subplot_titles=["Portfolio Value (¥)", "Drawdown (%)"],
    vertical_spacing=0.08,
)

fig.add_trace(
    go.Scatter(x=eq.index, y=eq["portfolio_value"],
               name="Portfolio", line=dict(color="#4fc3f7", width=2)),
    row=1, col=1
)

# Drawdown
rolling_max = eq["portfolio_value"].cummax()
drawdown = (eq["portfolio_value"] - rolling_max) / rolling_max * 100
fig.add_trace(
    go.Scatter(x=eq.index, y=drawdown, name="Drawdown",
               fill="tozeroy", line=dict(color="#ef5350", width=1),
               fillcolor="rgba(239,83,80,0.2)"),
    row=2, col=1
)

# Mark trade entries
if not result.trade_log.empty:
    entries = result.trade_log[
        result.trade_log["action"].isin(["BUY", "SHORT_OPEN"]) &
        (result.trade_log["leg"] == "A")
    ].copy()
    entries["date"] = pd.to_datetime(entries["date"])
    entry_vals = eq["portfolio_value"].reindex(entries["date"], method="nearest")
    fig.add_trace(
        go.Scatter(
            x=entries["date"], y=entry_vals.values,
            mode="markers",
            marker=dict(symbol="triangle-up", size=10, color="#66bb6a"),
            name="Trade Entry"
        ),
        row=1, col=1
    )

fig.update_layout(
    height=600, template="plotly_dark",
    title=f"Equity Curve — {cfg.ts_code_a} / {cfg.ts_code_b}",
    legend=dict(orientation="h", y=1.02)
)
fig.show()

## Chart 2 — Spread & Z-score with Trade Signals

In [ ]:
sp = result.spread_df

fig = make_subplots(
    rows=3, cols=1,
    shared_xaxes=True,
    row_heights=[0.35, 0.40, 0.25],
    subplot_titles=["Adjusted Close Prices (rebased to 100)", "Z-Score of Spread", "Rolling Hedge Ratio (β)"],
    vertical_spacing=0.07,
)

# Rebased prices
base_a = sp["adj_close_a"].iloc[0]
base_b = sp["adj_close_b"].iloc[0]
fig.add_trace(go.Scatter(x=sp.index, y=sp["adj_close_a"] / base_a * 100,
                         name=cfg.ts_code_a, line=dict(color="#4fc3f7")), row=1, col=1)
fig.add_trace(go.Scatter(x=sp.index, y=sp["adj_close_b"] / base_b * 100,
                         name=cfg.ts_code_b, line=dict(color="#ffb74d")), row=1, col=1)

# Z-score
fig.add_trace(go.Scatter(x=sp.index, y=sp["z_score"],
                         name="Z-Score", line=dict(color="#ce93d8", width=1.5)), row=2, col=1)

# Threshold lines
for level, color, dash in [
    (cfg.entry_z,     "#66bb6a", "dash"),
    (-cfg.entry_z,    "#66bb6a", "dash"),
    (cfg.exit_z,      "#ffffff", "dot"),
    (cfg.stop_loss_z, "#ef5350", "longdash"),
    (-cfg.stop_loss_z,"#ef5350", "longdash"),
]:
    fig.add_hline(y=level, line_dash=dash, line_color=color, opacity=0.6, row=2, col=1)

# Trade signals on z-score panel
if not result.trade_log.empty:
    tlog = result.trade_log.copy()
    tlog["date"] = pd.to_datetime(tlog["date"])
    a_trades = tlog[tlog["leg"] == "A"]

    for _, row_t in a_trades.iterrows():
        d = row_t["date"]
        if d not in sp.index:
            continue
        z_val = sp.loc[d, "z_score"] if d in sp.index else None
        if z_val is None:
            continue
        action = row_t["action"]
        color = "#66bb6a" if action in ("BUY", "SHORT_OPEN") else "#ef5350"
        symbol = "triangle-up" if action in ("BUY", "SHORT_OPEN") else "triangle-down"
        fig.add_trace(
            go.Scatter(x=[d], y=[z_val], mode="markers",
                       marker=dict(symbol=symbol, size=9, color=color),
                       showlegend=False),
            row=2, col=1
        )

# Hedge ratio
fig.add_trace(go.Scatter(x=sp.index, y=sp["hedge_ratio"],
                         name="β", line=dict(color="#80cbc4", width=1.5)), row=3, col=1)

# Mark drift alerts
for alert in result.drift_alerts:
    d = pd.to_datetime(alert["date"])
    if d in sp.index:
        fig.add_vline(x=d, line_color="#ff7043", line_dash="dot", opacity=0.7, row=3, col=1)

fig.update_layout(
    height=720, template="plotly_dark",
    title=f"Spread Analysis — {cfg.ts_code_a} / {cfg.ts_code_b}",
    legend=dict(orientation="h", y=1.02),
)
fig.show()

## Chart 3 — Trade Log

In [ ]:
if result.trade_log.empty:
    print("No trades executed.")
else:
    tlog = result.trade_log.copy()
    tlog["date"] = pd.to_datetime(tlog["date"])

    # Colour-coded table
    action_colors = {
        "BUY":         "#c8e6c9",
        "SELL":        "#ffcdd2",
        "SHORT_OPEN":  "#fff9c4",
        "SHORT_CLOSE": "#e1bee7",
    }
    tlog["date_str"] = tlog["date"].dt.strftime("%Y-%m-%d")
    tlog["notional"] = tlog["notional"].round(0).astype(int)
    tlog["transaction_cost"] = tlog["transaction_cost"].round(2)

    display(tlog[["date_str", "leg", "ts_code", "action", "shares", "price",
                   "notional", "transaction_cost", "note"]]
            .rename(columns={"date_str": "date"})
            .style.applymap(
                lambda v: f"background-color: {action_colors.get(v, 'transparent')}",
                subset=["action"]
            )
            .set_caption(f"Full Trade Log ({len(tlog)} rows)"))

    # PnL bar chart per round trip
    entries = tlog[tlog["action"].isin(["BUY", "SHORT_OPEN"]) & (tlog["leg"] == "A")].reset_index(drop=True)
    exits   = tlog[tlog["action"].isin(["SELL", "SHORT_CLOSE"]) & (tlog["leg"] == "A")].reset_index(drop=True)
    n = min(len(entries), len(exits))

    if n > 0:
        pnls = []
        labels = []
        for j in range(n):
            e = entries.iloc[j]
            x = exits.iloc[j]
            # A-leg only PnL for illustration; full PnL is in portfolio curve
            if e["action"] == "BUY":
                p = x["notional"] - e["notional"] - e["transaction_cost"] - x["transaction_cost"]
            else:
                p = e["notional"] - x["notional"] - e["transaction_cost"] - x["transaction_cost"]
            pnls.append(p)
            labels.append(f"{e['date_str']} → {x['date_str']}")

        colors = ["#66bb6a" if p > 0 else "#ef5350" for p in pnls]
        fig2 = go.Figure(go.Bar(
            x=labels, y=pnls,
            marker_color=colors,
            text=[f"¥{p:,.0f}" for p in pnls],
            textposition="auto"
        ))
        fig2.update_layout(
            title="Round-Trip PnL per Trade (A-leg, gross)",
            template="plotly_dark",
            xaxis_tickangle=-45,
            yaxis_title="PnL (¥)",
            height=400,
        )
        fig2.show()

## Export results

In [ ]:
pair_tag = f"{cfg.ts_code_a}_{cfg.ts_code_b}"
result.equity_curve.to_csv(f"equity_curve_{pair_tag}.csv")
result.spread_df.to_csv(f"spread_{pair_tag}.csv")
if not result.trade_log.empty:
    result.trade_log.to_csv(f"trade_log_{pair_tag}.csv", index=False)
pd.DataFrame([result.stats]).T.to_csv(f"stats_{pair_tag}.csv")
print("Exported equity curve, spread, trade log, and stats CSVs.")